# Практическое занятие: Нормализация текста
## Кейс: Нормализация на примере текста Л.Н. Толстого

В этом ноутбуке реализован сквозной пайплайн нормализации текста для русского и английского языков:
1. **Нижний регистр и очистка** от пунктуации.
2. **Удаление стоп-слов** (на базе корпусов NLTK).
3. **Стемминг** (выделение псевдоосновы).
4. **Лемматизация** (приведение к словарной канонической форме).

In [1]:
%pip install nltk pymorphy3 pandas

  Using cached nltk-3.10.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached pymorphy3-2.0.6-py3-none-any.whl.metadata (2.4 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached dawg2_python-0.9.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl.metadata (2.0 kB)
Using cached nltk-3.10.0-py3-none-any.whl (1.7 MB)
Using cached pymorphy3-2.0.6-py3-none-any.whl (53 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 11.5 MB/s  0:00:00eta 0:00:01
Using cached dawg2_python-0.9.0-py3-none-any.whl (9.3 kB)
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl (8.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [nltk]6/7 [nltk]s]
Note: you may need to restart the kernel to use updated packages.


In [10]:
import nltk

nltk.download('stopwords')
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

[nltk_data] Downloading package stopwords to /Users/ksrve/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Очистка текста и приведение к нижнему регистру

In [15]:
import re

text_ru = "«Что это? я падаю? у меня ноги подкашиваются», — подумал он и упал на спину."
text_en = "The quick brown fox jumps over the lazy dog, while listening to NLP lectures!"

print("RU Исходный:", text_ru)
print("EN Исходный:", text_en)

def clean_and_lowercase(text):
    # Приведение к нижнему регистру
    text_low = text.lower()
    # Удаление знаков препинания и символов
    text_clean = re.sub(r'[^a-zA-Zа-яА-ЯёЁ\s]', '', text_low)
    return text_clean

cleaned_ru = clean_and_lowercase(text_ru)
cleaned_en = clean_and_lowercase(text_en)

print("RU Очищенный:", cleaned_ru)
print("EN Очищенный:", cleaned_en)

RU Исходный: «Что это? я падаю? у меня ноги подкашиваются», — подумал он и упал на спину.
EN Исходный: The quick brown fox jumps over the lazy dog, while listening to NLP lectures!
RU Очищенный: что это я падаю у меня ноги подкашиваются  подумал он и упал на спину
EN Очищенный: the quick brown fox jumps over the lazy dog while listening to nlp lectures


## Токенизация и удаление стоп-слов

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Функция для вычисления Word Shape
def get_word_shape(word):
    shape = []
    for char in word:
        if char.isupper():
            shape.append('X')
        elif char.islower():
            shape.append('x')
        elif char.isdigit():
            shape.append('d')
        else:
            shape.append(char) # Сохраняем дефисы, апострофы и спецсимволы
    return "".join(shape)

# Предположим, что на вход подаются очищенные строки (но еще сохраняющие оригинальный регистр)
# Для работы Word Shapes важно, чтобы регистр НЕ был приведен к нижнему на Шаге 1
cleaned_ru = "Кутузов ехал на лошади в 1805 году"
cleaned_en = "Kutuzov rode a horse in 1805"

print("RU Очищенный:", cleaned_ru)
print("EN Очищенный:", cleaned_en)

stop_words_ru = set(stopwords.words('russian'))
stop_words_en = set(stopwords.words('english'))

# Разбиваем строки на отдельные слова-токены
tokens_ru = word_tokenize(cleaned_ru)
tokens_en = word_tokenize(cleaned_en)

# Вычисляем Word Shapes ДО приведения к нижнему регистру и фильтрации
# (Позволяет понять структуру исходных слов)
shapes_ru = [get_word_shape(w) for w in tokens_ru]
shapes_en = [get_word_shape(w) for w in tokens_en]

print("RU Формы слов (Shapes):", shapes_ru)
print("EN Формы слов (Shapes):", shapes_en)

# Убираем незначимые части речи (стоп-слова)
# Чтобы стоп-слова сработали корректно, проверяем их в нижнем регистре (.lower())
filtered_ru = [w for w in tokens_ru if w.lower() not in stop_words_ru]
filtered_en = [w for w in tokens_en if w.lower() not in stop_words_en]

print("RU Фильтрация:", filtered_ru)
print("EN Фильтрация:", filtered_en)


RU Очищенный: Кутузов ехал на лошади в 1805 году
EN Очищенный: Kutuzov rode a horse in 1805
RU Формы слов (Shapes): ['Xxxxxxx', 'xxxx', 'xx', 'xxxxxx', 'x', 'dddd', 'xxxx']
EN Формы слов (Shapes): ['Xxxxxxx', 'xxxx', 'x', 'xxxxx', 'xx', 'dddd']
RU Фильтрация: ['Кутузов', 'ехал', 'лошади', '1805', 'году']
EN Фильтрация: ['Kutuzov', 'rode', 'horse', '1805']


## Стемминг (Stemming)
Грубое усечение словоформ до их основы.

Porter Stemmer (Алгоритм Портера)

Разработан Мартином Портером в 1980 году. Это классический алгоритм, представляющий собой линейную последовательность шагов, на каждом из которых применяются правила продукций вида:

$$\text{Condition} \rightarrow \text{Action}$$

### Алгоритмическая структура
Слово представляется в виде чередования гласных ($V$) и согласных ($C$). Любое слово можно записать в обобщенной форме:

$$[C](V C)^m[V]$$

где:
* $m$ — **мера слова** (число комбинаций $VC$).
* $[ ]$ — обозначает необязательность элемента.

Правила удаления суффиксов жестко завязаны на значение $m$. Например, правило для суффикса `-eed`:

$$(m > 0) \text{ EED} \rightarrow \text{ EE}$$

* Для слова *agreed* ($m=1$, основа *agr-*, $VC$): $\text{agreed} \rightarrow \text{agree}$
* Для слова *feed* ($m=0$, основа $F$, нет $VC$): изменений нет.
  
### Ограничения Porter Stemmer
1. **Только английский язык:** Алгоритм жестко закодирован под грамматику английского языка.
2. **Перерегулирование (Overstemming):** Избыточное усечение слов, приводящее к потере смысла.
   $$\text{organization} \rightarrow \text{organ}, \quad \text{organ} \rightarrow \text{organ}$$
3. **Недорегулирование (Understemming):** Неспособность связать формы одного слова.
   $$\text{alumnus} \rightarrow \text{alumnu}, \quad \text{alumni} \rightarrow \text{alumni}$$


**Snowball** — это не просто один алгоритм, а разработанный Мартином Портером язык программирования для создания стеммеров. Реализованный на нем алгоритм для английского языка называют **Porter2**.

### Основные улучшения и математика зон
В Snowball вводится понятие текстовых зон $R_1$ и $R_2$, которые динамически определяют контекст применения правил.

Пусть слово $S$ состоит из последовательности символов. Зона $R_1$ — это область слова после первой комбинации "гласный-согласный" ($VC$). Зона $R_2$ — это область после первой комбинации $VC$ внутри зоны $R_1$.

$$S = \underbrace{c_1 c_2 \dots v_1 c_3}_{R_1 \text{ старт}} \dots v_2 c_4 \dots$$

**Пример разделения для слова *beautiful*:**
* $R_1$: `tiful` (все, что после первого $VC$, то есть после `beau`)
* $R_2$: `ful` (все, что после первого $VC$ внутри $R_1$, то есть после `ti`)

Правила в Snowball применяются только в том случае, если суффикс полностью находится внутри зоны $R_1$ или $R_2$. Это уберегает короткие слова от ложного усечения.

### Преимущества Snowball Stemmer
* **Многоязычность:** Существуют официальные алгоритмы для русского (`RussianStemmer`), испанского, немецкого, французского и других языков.
* **Повышенная точность:** Исправлены проблемы с обработкой суффиксов вроде `-ly` (например, $\text{fairly} \rightarrow \text{fair}$, в отличие от Porter, где $\text{fairly} \rightarrow \text{fairli}$).
* **Скорость:** Скомпилированный код на языке Snowball работает быстрее за счет оптимизации переходов состояний.



In [8]:
from nltk.stem import PorterStemmer, SnowballStemmer

stemmer_ru = SnowballStemmer('russian')
stemmer_en = PorterStemmer()

stemmed_ru = [stemmer_ru.stem(w) for w in filtered_ru]
stemmed_en = [stemmer_en.stem(w) for w in filtered_en]

print("RU Стемминг:", stemmed_ru)
print("EN Стемминг:", stemmed_en)

RU Стемминг: ['эт', 'пада', 'ног', 'подкашива', 'подума', 'упа', 'спин']
EN Стемминг: ['quick', 'brown', 'fox', 'jump', 'lazi', 'dog', 'listen', 'nlp', 'lectur']


## Лемматизация (Lemmatization)
Приведение слов к полноценной словарной форме

In [11]:
import pymorphy3

# Морфологический разбор для русского языка
morph = pymorphy3.MorphAnalyzer()
lemmatized_ru = [morph.parse(w)[0].normal_form for w in filtered_ru]

# Лемматизатор для английского языка
from nltk.stem import WordNetLemmatizer
lemmatizer_en = WordNetLemmatizer()
lemmatized_en = [lemmatizer_en.lemmatize(w) for w in filtered_en]

print("RU Лемматизация:", lemmatized_ru)
print("EN Лемматизация:", lemmatized_en)

RU Лемматизация: ['это', 'падать', 'нога', 'подкашиваться', 'подумать', 'упасть', 'спина']
EN Лемматизация: ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog', 'listening', 'nlp', 'lecture']


## Сравнительная таблица результатов

In [13]:
import pandas as pd

print("Русский язык:")
df_ru = pd.DataFrame({'Токен': filtered_ru, 'Стемминг': stemmed_ru, 'Лемматизация': lemmatized_ru})
display(df_ru)

print("\nEnglish:")
df_en = pd.DataFrame({'Token': filtered_en, 'Stemming': stemmed_en, 'Lemmatization': lemmatized_en})
display(df_en)

Русский язык:


,Токен,Стемминг,Лемматизация
0,это,эт,это
1,падаю,пада,падать
2,ноги,ног,нога
3,подкашиваются,подкашива,подкашиваться
4,подумал,подума,подумать
5,упал,упа,упасть
6,спину,спин,спина



English:


,Token,Stemming,Lemmatization
0,quick,quick,quick
1,brown,brown,brown
2,fox,fox,fox
3,jumps,jump,jump
4,lazy,lazi,lazy
5,dog,dog,dog
6,listening,listen,listening
7,nlp,nlp,nlp
8,lectures,lectur,lecture
